In [ ]:
# ============================================================
# CONFIGURATION - every path comes from config/paths.py, the single
# source of truth. Override cluster locations with the MUSICA_ENV_*
# environment variables documented there. Do not hard-code paths here.
# ============================================================
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
_ROOT = next(p for p in [_here, *_here.parents]
             if (p / 'config' / 'paths.py').exists())
sys.path.insert(0, str(_ROOT))
import config  # also puts functions/ on sys.path
from config import paths as P


This script works on addressing need to get estimates of % O3 produced versus imported

In [ ]:
import os
import glob
import fnmatch

import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point, Polygon

import xarray as xr
import numpy as np

import matplotlib.pyplot as plt # Core library for plotting
import matplotlib.cm as cm # To use different colormaps
import cartopy.crs as ccrs # For map projection
import seaborn as sns # boxplot


In [ ]:
# !pip install geopandas


In [ ]:
import sys
sys.path.insert(0,f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/')

from Plot_2D import Plot_2D # To draw a map

SCRIP_CONUS = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne0CONUS_ne30x8_np4_SCRIP.nc'
SCRIP_ne30 = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons.nc'

In [ ]:
def molecules_to_kg_per_m2_per_s(molecules_per_cm2_per_s, molecular_weight):
    """
        Read in emissions data in [molecules cm-2 s-1], and molecular_weight in [g/mole]
        Return emissions in [kg m-2 s-1]    
    """
    # Constants
    avogadro_number = 6.022e23  # molecules per mole
    m2_to_cm2 = 1e4  # square meters to square centimeters 
    kg_to_g = 1e3

    # Convert 
    kg_per_m2_per_s = molecules_per_cm2_per_s*(1/avogadro_number)*molecular_weight*(1/kg_to_g)*(m2_to_cm2)

    return kg_per_m2_per_s

# # Example usage
# molecules_per_cm2_per_s = 1  # for example, 1e18 molecules per cm^2 per second
# molecular_weight = 30  # molecular weight of water in g/mol

# result = molecules_to_kg_per_m2_per_s(molecules_per_cm2_per_s, molecular_weight)
# print("Result:", result, "kg/m^2/s")


setunit = r'$kg$ $m^{-2}s^{-1}$'

In [ ]:
### Geo information for boundaries
import geopandas as gpd

# Path to the shapefile (adjust if the file is inside a folder)
shapefile_path = f'{P.HOME_ROOT}/HelpfulFiles/world-administrative-boundaries/world-administrative-boundaries.shp'

# Read shapefile directly
gdf = gpd.read_file(shapefile_path)

# # Preview
# print(gdf.columns)
# print(gdf.head())

# Adjust based on the actual column names (use print(gdf.columns) to confirm)
us_boundary = gdf[(gdf['status'] == 'Member State')&(gdf['name'] == 'United States of America')]  # or 'USA', depending on the file

# Create a GeoDataFrame
us_gdf = gpd.GeoDataFrame(us_boundary, geometry='geometry')

from shapely.geometry import Point
import numpy as np
import xarray as xr


# Remove emissions over CONUS Box-land

In [ ]:
### Read in NEI emissions data
CAMS_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_copied2021_c20241011/'

CONUSlandMasked_diri = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_CONUSlandMasked_copied2021_c20250526/'
# NYCMasked_diri = '{P.CESM22_ROOT}/CAMS_withCONUS2017NEI/NYCMasked_MappedSpecies_GLOB_Merged_CAMS_NEI_ne0CONUSne30x8_JulyMean/' # where to put the Masked files


In [ ]:
# Search for files that match the pattern

import glob
import os

# Directory where files are stored
# CAMS_diri = '/path/to/your/files'  # Make sure this is correctly defined

# Updated pattern with wildcard
pattern = os.path.join(CAMS_diri, 'CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_*.nc')

# Get list of matching files in full path
CAMS_v51_file_list = glob.glob(pattern)
# print(CAMS_v51_file_list)

# Extract only the filenames
CAMS_v51_file_names = [os.path.basename(f) for f in CAMS_v51_file_list]

print(CAMS_v51_file_names[:3])


In [ ]:
# CAMS_v51_file_names[0].split('_')[4]

spc_ls = []
for spcIdx in range(len(CAMS_v51_file_names)):
    spc_name = CAMS_v51_file_names[spcIdx].split('_')[4]
    spc_ls.append(spc_name)

In [ ]:
import re
# Extract species name between 'ne30np4_' and '_c20210423'
species_names = [
    re.search(r'ne30np4_(.+?)_c20210423', f).group(1)
    for f in CAMS_v51_file_names
]
species_names = sorted(species_names)
print(species_names)

In [ ]:
# CAMS_v51_file_list

In [ ]:
spc_fileOUTpath = f'{CAMS_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'

In [ ]:
Abs_rangeMax_dic = {'NO':7e-11,
                  'CO':5e-10,
                  'SO2':3e-11, 
                  'bc_a4':3e-12,
                  'NH3':2e-11,
                  'C2H2':2e-12, 
                  'C2H4':5e-12, # ethene
                  'C2H5OH':5e-11,
                  'C2H6':6e-12, # ethane
                  'C3H6':2e-11, 
                  'C3H8':2e-12, 
                  'CH3OH':2e-12,
                  'CH3CHO':2e-12, # 
                  'CH3COCH3':2e-12, 
                  'MEK':2e-12,
                    
                # Second panel
                  'BENZENE':3e-12, 
                  'TOLUENE':7e-12,
                  'BIGENE':4e-12,   
                  'MTERP':5e-13, 
                  'ISOP':8e-14, 
                  'CH2O':3e-12, # formaldehyde
                  'SVOC':6e-12,
                   }

## Note that the magnitude for some VOC species is much lower compared to their biogenic emissions
# MTERP: 4.90e-13 | 2.14e-13 (anthrop) compared to 
# ISOP: 8.37e-14 | 7.34e-14 (anthrop) compared to 

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'DejaVu Sans'  # Change to a font that supports the superscript characters

def get_superscript(magnitude):
    return r'$10^{' + str(magnitude) + '}$'

### Remove for a list of species files

In [ ]:
# ### Create the Masked array and save to nc file | use NO
# spc = 'NO'
# spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'
# # Open Xarray dataset
# spci_ds = xr.open_dataset(spc_fileINpath)

# # MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
# lat = spci_ds['lat'].values
# lon = spci_ds['lon'].values
# Adjustedlons = np.where(lon >= 180, lon - 360, lon)
# # Create a DataArray for the adjusted lon
# Adjustedlons_da = xr.DataArray(
#                                 Adjustedlons,  
#                                 dims=('ncol'),
#                                 coords={'ncol': spci_ds.ncol.values} 
#                                 )

# # Add the DataArray back to the dataset
# spci_ds['Adjustedlons'] = Adjustedlons_da

# ### To process for all months for the given period (reduce the size of processed dataset)
# Timei = '2022-07-31T00:00:00.000000000'
# MonthiMean_ds = spci_ds.sel(time=Timei)
# latitudes = MonthiMean_ds.lat.values
# longitudes = MonthiMean_ds.Adjustedlons.values
# # MonthiMean_ds

# # Step 1: Define CONUS bounding box
# lon_lefti = -66.95   # Eastern edge
# lon_righti = -125.0  # Western edge
# lat_boti = 24.4      # Southern edge
# lat_upi = 49.5       # Northern edge

# # Step 2: Create a combined mask
# mask = np.zeros(len(lat), dtype=bool)

# for geom in gdf['geometry']:
#     for i in range(len(lat)):
#         lon = Adjustedlons[i]
#         lati = lat[i]
#         point = Point(lon, lati)

#         # Only mask if inside CONUS bounding box and also within the geometry
#         in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
#         if in_box and geom.contains(point):
#             mask[i] = True

# # Step 3: Convert to xarray mask
# mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# # Step 4: Apply the mask to remove CONUS land pixels
# masked_ds = MonthiMean_ds.where(~mask_da, drop=False)
# # sum_masked = masked_ds['sum']

# ### Can probably save this masked array to use later 
# maskfilePath = '{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_CONUSlandMaskedFalse.nc'
# # Save mask_da to a NetCDF file
# mask_da.to_netcdf(maskfilePath)
# print('Save to:', maskfilePath)

In [ ]:
ne30maskfilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_CONUSlandMaskedFalse.nc'
# Read the saved NetCDF file
mask_da_loaded = xr.open_dataarray(ne30maskfilePath)

In [ ]:
# masked_ds = MonthiMean_ds.where(~mask_da_loaded, drop=False)
# masked_ds

In [ ]:
targetspc_ls = ['ISOP', 'NO']
timeslice_start = '2021-11-30T00:00:00.000000000'
timeslice_end = '2022-11-30T00:00:00.000000000'

for spc in targetspc_ls:
    spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'
    
    # Open Xarray dataset
    spci_ds = xr.open_dataset(spc_fileINpath)

    # MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
    lat = spci_ds['lat'].values
    lon = spci_ds['lon'].values
    Adjustedlons = np.where(lon >= 180, lon - 360, lon)
    # Create a DataArray for the adjusted lon
    Adjustedlons_da = xr.DataArray(
                                    Adjustedlons,  
                                    dims=('ncol'),
                                    coords={'ncol': spci_ds.ncol.values} 
                                    )

    # Add the DataArray back to the dataset
    spci_ds['Adjustedlons'] = Adjustedlons_da
    
    ### Use the mask read in to remove pixels over CONUS-land, need to use zero instead of nan
    sum_masked = spci_ds['sum'].where(~mask_da_loaded, other=0)

    spci_ds['sum'] = sum_masked

    # Remove the 'Adjustedlons' variable from the dataset
    spci_ds = spci_ds.drop_vars('Adjustedlons')
    
    ### Save masked to the corresponding file
    spc_fileOUTpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'
    spci_ds.to_netcdf(spc_fileOUTpath)
    print('Save to:',spc_fileOUTpath)

In [ ]:
masked_ds

In [ ]:
testfile = f'{P.NCAR_COPIES_ROOT}/acom/MUSICA/emissions/cams/CAMS-GLOB-ANTv5.1/ne30np4/Y2022_CONUSlandMasked_copied2021_c20250526/CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_NO_c20210423_modified.nc'
test_ds = xr.open_dataset(testfile)

### Plotting
# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

# Quick plot to check
Timei = '2022-07-01T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne30)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = test_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = masked_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, 
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
testfile = f'{CAMS_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_NO_c20210423_modified.nc'
test_ds = xr.open_dataset(testfile)

### Plotting
# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

# Quick plot to check
Timei = '2022-07-01T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne30)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = test_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = masked_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, 
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
## Read back
for spc in targetspc_ls:
    spc_maskedINpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'
    masked_ds = xr.open_dataset(spc_maskedINpath)

    ### Plotting
    # get rangeMax
    rangeMax = Abs_rangeMax_dic[spc]

    ### Accounting for scientific notation
    # Get the magnitude in scientific notation, adjust for e-
    magnitude = int(np.floor(np.log10(abs(rangeMax))))
    # Format the magnitude as a superscript
    formatted_magnitude = get_superscript(str(magnitude))
    modified_rangeMax = rangeMax*(1/(10)**magnitude)
    
    # Quick plot to check
    Timei = '2022-07-01T00:00:00.000000000'

    setmap = 'viridis'

    """ds_mappedMUSICA (regridded to ne30)"""
    PlotRegion = 'CONUS'

    # scalefactor = 1e-11
    # formatted_label = f"{scalefactor:.0e}" 

    Plot_ar = masked_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

    vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
    Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
    longname = masked_ds['sum'].long_name
    rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

    ### Which map
    fig = plt.figure( figsize=(8,6) ) 
    # - ne30x8 regional refinement over CONUS|
    ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
    if PlotRegion=="CONUS":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, lon_range=[-140,-50], lat_range=[15,60],
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
    elif PlotRegion=="Global":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, 
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

    plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
masked_ds.time.values

In [ ]:
## Read back
for spc in targetspc_ls:
    spc_maskedINpath = f'{CONUSlandMasked_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'
    masked_ds = xr.open_dataset(spc_maskedINpath)

    ### Plotting
    # get rangeMax
    rangeMax = Abs_rangeMax_dic[spc]

    ### Accounting for scientific notation
    # Get the magnitude in scientific notation, adjust for e-
    magnitude = int(np.floor(np.log10(abs(rangeMax))))
    # Format the magnitude as a superscript
    formatted_magnitude = get_superscript(str(magnitude))
    modified_rangeMax = rangeMax*(1/(10)**magnitude)
    
    # Quick plot to check
    Timei = '2022-07-01T00:00:00.000000000'

    setmap = 'viridis'

    """ds_mappedMUSICA (regridded to ne30)"""
    PlotRegion = 'Global'

    # scalefactor = 1e-11
    # formatted_label = f"{scalefactor:.0e}" 

    Plot_ar = masked_ds.sel(time=Timei,method='nearest')['sum'].values#*scalefactor

    vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
    Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
    longname = masked_ds['sum'].long_name
    rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

    ### Which map
    fig = plt.figure( figsize=(8,6) ) 
    # - ne30x8 regional refinement over CONUS|
    ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
    if PlotRegion=="CONUS":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, lon_range=[-140,-50], lat_range=[15,60],
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
    elif PlotRegion=="Global":
        im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
                cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
                unit=f'{formatted_magnitude}{Plot_unit}',
                state=True, 
                  grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

    plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

### Test for Nitric Oxide Emissions (NO) | 2-D and only contains 'sum' sector

In [ ]:
spc = 'NO'
spc_fileINpath = f'{CAMS_diri}CAMS-GLOB-ANT_v5.1_2000-2021_ne30np4_{spc}_c20210423_modified.nc'

# get rangeMax
rangeMax = Abs_rangeMax_dic[spc]

### Accounting for scientific notation
# Get the magnitude in scientific notation, adjust for e-
magnitude = int(np.floor(np.log10(abs(rangeMax))))
# Format the magnitude as a superscript
formatted_magnitude = get_superscript(str(magnitude))
modified_rangeMax = rangeMax*(1/(10)**magnitude)

In [ ]:
# Open Xarray dataset
spci_ds = xr.open_dataset(spc_fileINpath)

# MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
lat = spci_ds['lat'].values
lon = spci_ds['lon'].values
Adjustedlons = np.where(lon >= 180, lon - 360, lon)
# Create a DataArray for the adjusted lon
Adjustedlons_da = xr.DataArray(
                                Adjustedlons,  
                                dims=('ncol'),
                                coords={'ncol': spci_ds.ncol.values} 
                                )

# Add the DataArray back to the dataset
spci_ds['Adjustedlons'] = Adjustedlons_da

In [ ]:
spci_ds

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 
MonthiMean_da = spci_ds.sel(time=Timei)['sum']

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
spci_ds.time.values

In [ ]:
timeslice_start = '2021-11-30T00:00:00.000000000'
timeslice_end = '2022-11-30T00:00:00.000000000'


seltimeMean_ds = spci_ds.sel(time=slice(timeslice_start,timeslice_end))
latitudes = seltimeMean_ds.lat.values
longitudes = seltimeMean_ds.Adjustedlons.values
# seltimeMean_ds

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Create a combined mask
mask = np.zeros(len(lat), dtype=bool)

for geom in gdf['geometry']:
    for i in range(len(lat)):
        lon = Adjustedlons[i]
        lati = lat[i]
        point = Point(lon, lati)

        # Only mask if inside CONUS bounding box and also within the geometry
        in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
        if in_box and geom.contains(point):
            mask[i] = True

# Step 3: Convert to xarray mask
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = seltimeMean_ds.where(~mask_da, drop=False)
sum_masked = masked_ds['sum']


In [ ]:
### Can probably save this masked array to use later 
savefilePath = f'{P.HOME_ROOT}/Scripts/CESM_analysis/functions/ne30np4_091226_pentagons_CONUSlandMaskedFalse.nc'
# Save mask_da to a NetCDF file
mask_da.to_netcdf(savefilePath)

In [ ]:
mask_da

In [ ]:
masked_ds = seltimeMean_ds.where(~mask_da, drop=False)
masked_ds

In [ ]:
### To process for all months for the given period (reduce the size of processed dataset)
Timei = '2022-07-31T00:00:00.000000000'

MonthiMean_ds = spci_ds.sel(time=Timei)
latitudes = MonthiMean_ds.lat.values
longitudes = MonthiMean_ds.Adjustedlons.values
# MonthiMean_ds

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Create a combined mask
mask = np.zeros(len(lat), dtype=bool)

for geom in gdf['geometry']:
    for i in range(len(lat)):
        lon = Adjustedlons[i]
        lati = lat[i]
        point = Point(lon, lati)

        # Only mask if inside CONUS bounding box and also within the geometry
        in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
        if in_box and geom.contains(point):
            mask[i] = True

# Step 3: Convert to xarray mask
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False)
sum_masked = masked_ds['sum']


In [ ]:
### To process for one month
Timei = '2022-07-31T00:00:00.000000000'

MonthiMean_ds = spci_ds.sel(time=Timei)
latitudes = MonthiMean_ds.lat.values
longitudes = MonthiMean_ds.Adjustedlons.values
# MonthiMean_ds

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Create a combined mask
mask = np.zeros(len(lat), dtype=bool)

for geom in gdf['geometry']:
    for i in range(len(lat)):
        lon = Adjustedlons[i]
        lati = lat[i]
        point = Point(lon, lati)

        # Only mask if inside CONUS bounding box and also within the geometry
        in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
        if in_box and geom.contains(point):
            mask[i] = True

# Step 3: Convert to xarray mask
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False)
sum_masked = masked_ds['sum']


In [ ]:
sum_masked

In [ ]:
JulyMeanNEIMerged_vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_CONUS, ax=ax,
                unit=formatted_magnitude,unit_size=30,title_size=40,colortick_size=30,
                 cmin=0.1, cmax=modified_rangeMax

In [ ]:
molecules_to_kg_per_m2_per_s

In [ ]:
# Quick plot to check

setmap =  'viridis' #'CMRmap_r'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = spci_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            # state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

setmap = 'viridis' #'CMRmap_r'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
vari_kg_per_m2_per_s = molecules_to_kg_per_m2_per_s(Plot_ar, spci_ds['sum'].molecular_weight)
Plot_unit = r'$kg$ $m^{-2}$ $s^{-1}$'
longname = spci_ds['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True ) 
elif PlotRegion=="Global":
    im = Plot_2D( vari_kg_per_m2_per_s*(1/(10)**magnitude), scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0.1, cmax=modified_rangeMax, cmap=setmap, 
            unit=f'{formatted_magnitude}{Plot_unit}',
            # state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15, twodecimal=True )   

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

### Trial

In [ ]:
MonthiMean_ds = spci_ds.sel(time=Timei)
latitudes = MonthiMean_ds.lat.values
longitudes = MonthiMean_ds.Adjustedlons.values
MonthiMean_ds

In [ ]:
### First select over land
import rasterio
from rasterio.features import rasterize
# **Load High-Resolution Land Shapefile**
land_shapefile = f"{P.HOME_ROOT}/HelpfulFiles/ne_10m_land/ne_10m_land.shp"
gdf_land = gpd.read_file(land_shapefile)

# select for land pixel only
# **Define Raster Transform (Corrected)**
transform = rasterio.transform.from_bounds(
    longitudes.min(), latitudes.max(),  # Bottom-left corner
    longitudes.max(), latitudes.min(),  # Top-right corner
    len(longitudes), len(latitudes)     # Grid size
)

# **Rasterize Land Mask (1=Land, 0=Water)**
land_mask = rasterize(
    [(geom, 1) for geom in gdf_land.geometry], 
    out_shape=(len(latitudes), len(longitudes)), 
    transform=transform, 
    fill=0, 
    all_touched=True,  # Ensures land edges are included
    dtype=np.uint8
)

land_mask_xr = xr.DataArray(
    land_mask, 
    dims=["lat", "Adjustedlons"], 
    coords={"lat": MonthiMean_ds["lat"], "Adjustedlons": MonthiMean_ds["Adjustedlons"]}
)

# **Apply Mask to Dataset**
MonthiMean_da_land = MonthiMean_da.where(land_mask_xr)

In [ ]:
gdf_land

In [ ]:
### Mask values within the CONUS box (rectangular lat-lon box)
fileregion = 'CONUS_refined'

# Refined lat-lon bounding box for CONUS
lon_lefti = -66.95   # Eastern edge (near Maine)
lon_righti = -125.0  # Western edge (near California)
lat_boti = 24.4      # Southern edge (near southern Florida)
lat_upi = 49.5       # Northern edge (U.S.–Canada border)


# Define the mask for the bounding box
mask_inside = (
    (MonthiMean_ds['Adjustedlons'] >= lon_righti) &
    (MonthiMean_ds['Adjustedlons'] <= lon_lefti) &
    (MonthiMean_ds['lat'] >= lat_boti) &
    (MonthiMean_ds['lat'] <= lat_upi)
)

# Invert the mask to get data OUTSIDE the box
mask_outside = ~mask_inside

# Apply the mask to get data outside the region
outside_region_VCDds = MonthiMean_ds.where(mask_outside)


In [ ]:
outside_region_VCDds

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = outside_region_VCDds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = outside_region_VCDds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
### To remove only land pixels over the CONUS (keep everythingelse)
# Remove if NOT over water & within the CONUS box 

# Step 1: Pixels outside the CONUS box
mask_outside = ~mask_inside

# Step 2: Pixels inside CONUS box but over water
mask_inside_water = mask_inside & (~land_mask)

# Step 3: Combine the two
final_mask = mask_outside | mask_inside_water

# Step 4: Apply the mask
outside_water_or_outsideCONUS_ds = MonthiMean_ds.where(final_mask)


In [ ]:
import geopandas as gpd

# Path to the shapefile (adjust if the file is inside a folder)
shapefile_path = f'{P.HOME_ROOT}/HelpfulFiles/world-administrative-boundaries/world-administrative-boundaries.shp'

# Read shapefile directly
gdf = gpd.read_file(shapefile_path)

# Preview
print(gdf.columns)
print(gdf.head())


In [ ]:
# Adjust based on the actual column names (use print(gdf.columns) to confirm)
us_boundary = gdf[(gdf['status'] == 'Member State')&(gdf['name'] == 'United States of America')]  # or 'USA', depending on the file

# Create a GeoDataFrame
us_gdf = gpd.GeoDataFrame(us_boundary, geometry='geometry')

In [ ]:
us_gdf

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

# Plot the GeoDataFrame
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})
# gdf.plot(ax=ax, color='tab:blue', edgecolor='black')
us_gdf.plot(ax=ax, color='white', edgecolor='tab:blue')

# # Set the extent to the bounding box coordinates
# ax.set_extent([-74.5, -73.5, 40.4, 41], crs=ccrs.PlateCarree())

# Customize the plot (optional)
ax.set_title('U.S. Boundary')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')

# Show the plot
plt.show()

In [ ]:
# # Create a mask for the dataset
# mask = np.zeros(len(lat), dtype=bool)

# # Iterate over the geometries and update the mask
# points_within_bounds = []
# for geom in gdf['geometry']:
#     for i in range(len(lat)):
#         point = Point(Adjustedlons[i], lat[i])
#         if geom.contains(point):
#             points_within_bounds.append((Adjustedlons[i], lat[i]))
#             mask[i] = True

# # Convert mask to DataArray
# mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# # Apply the mask to the dataset
# masked_ds = MonthiMean_ds.where(mask_da, drop=True)

# # Apply the mask to the 'sum' variable
# # sum_masked = spci_ds['sum'].where(~mask_da, other=np.nan)
# sum_masked = spci_ds['sum'].where(~mask_da, other=0)

In [ ]:
from shapely.geometry import Point
import numpy as np
import xarray as xr

# Step 1: Define CONUS bounding box
lon_lefti = -66.95   # Eastern edge
lon_righti = -125.0  # Western edge
lat_boti = 24.4      # Southern edge
lat_upi = 49.5       # Northern edge

# Step 2: Create a combined mask
mask = np.zeros(len(lat), dtype=bool)

for geom in gdf['geometry']:
    for i in range(len(lat)):
        lon = Adjustedlons[i]
        lati = lat[i]
        point = Point(lon, lati)

        # Only mask if inside CONUS bounding box and also within the geometry
        in_box = (lon >= lon_righti) & (lon <= lon_lefti) & (lati >= lat_boti) & (lati <= lat_upi)
        if in_box and geom.contains(point):
            mask[i] = True

# Step 3: Convert to xarray mask
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False) # do not drop to keep the same size

# Optionally apply it to just one variable
sum_masked = masked_ds['sum']


In [ ]:
# Step 4: Apply the mask to remove CONUS land pixels
masked_ds = MonthiMean_ds.where(~mask_da, drop=False)
sum_masked = masked_ds['sum']

In [ ]:
MonthiMean_ds

In [ ]:
masked_ds

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Quick plot to check

Timei = '2022-07-31T00:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'Global'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = masked_ds['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_ne30, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

### Continue to process for all target files

In [ ]:
# get a list of species name I have updated 
MappedSpecies_dir = hourlyNEI_diri
DateOfModify = 'c20231113' #'c20231218';'c20231113'

# Search for files that match the pattern
pattern = '*NEI2017Merged_20181MJuly_CAMSv5.1*'+DateOfModify+'*'
Out_file_list = [file for file in glob.glob(os.path.join(MappedSpecies_dir, pattern)) if fnmatch.fnmatch(os.path.basename(file), pattern)]


In [ ]:
len(Out_file_list)

updated_spcs = []

for fileidx in range(len(Out_file_list)):
    filename = Out_file_list[fileidx].split('/')[-1]
    # Find the starting and ending indexes for the species name
    start_index = filename.find('ne0CONUSne30x8_') + len('ne0CONUSne30x8_')
    end_index = filename.find(DateOfModify)
    # Extract the species name using the start and end indexes
    species_name = filename[start_index:end_index-1]
    # print(species_name) 
    updated_spcs.append(species_name)
    
print(f'Number of updated species: {len(updated_spcs)}')

In [ ]:
### Mask emissions within NYC boundary zero

# Read in each file in the format 
# for spc in updated_spcs[:1]:
for spc in updated_spcs:
    # FileInPath = f'{JulyMeanNEI_diri}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_{spc}_{DateOfModify}.nc'
    FileInPath = f'{hourlyNEI_diri}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_{spc}_c20231113.nc'
    # Where to put the output
    FileOutPath = f'{NYCMasked_diri}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_{spc}_{DateOfModify}.nc'
    # check file existence
    if os.path.exists(FileInPath)==False:
        print(f"{FileInPath} missing.")
    else:
        # Proceed
        print( 'Write July mean: ', spc )
        print( '************************************************************************' )
        # Read in Xarray dataset
        spci_ds = xr.open_dataset(FileInPath)

        # MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
        lat = spci_ds['lat'].values
        lon = spci_ds['lon'].values
        Adjustedlons = np.where(lon >= 180, lon - 360, lon)
        # Create a DataArray for the adjusted lon
        Adjustedlons_da = xr.DataArray(
                                        Adjustedlons,  
                                        dims=('ncol'),
                                        coords={'ncol': spci_ds.ncol.values} 
                                        )

        # Add the DataArray back to the dataset
        spci_ds['Adjustedlons'] = Adjustedlons_da

        # Create a mask for the dataset
        mask = np.zeros(len(lat), dtype=bool)

        # Iterate over the geometries and update the mask
        points_within_bounds = []
        for geom in gdf['geometry']:
            for i in range(len(lat)):
                point = Point(Adjustedlons[i], lat[i])
                if geom.contains(point):
                    points_within_bounds.append((Adjustedlons[i], lat[i]))
                    mask[i] = True

        # Convert mask to DataArray
        mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

        # Apply the mask to the dataset
        masked_ds = spci_ds.where(mask_da, drop=True)

        # Apply the mask to the 'sum' variable
        # sum_masked = spci_ds['sum'].where(~mask_da, other=np.nan)
        sum_masked = spci_ds['sum'].where(~mask_da, other=0)

        spci_ds['sum'] = sum_masked
        
        # Remove the 'Adjustedlons' variable from the dataset
        spci_ds = spci_ds.drop_vars('Adjustedlons')

        ### save file
        spci_ds.to_netcdf(FileOutPath)
        print('Saved to:',FileOutPath)

In [ ]:
sum_masked

In [ ]:
# updated_spcs

### Draft

In [ ]:
for spc in updated_spcs:
    FileInPath = f'{MappedSpecies_dir}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_{spc}_{DateOfModify}.nc'
    

In [ ]:
# Open Xarray dataset
spci_ds = xr.open_dataset(FileInPath)

# MUSICA lons goes from 0-360, convert it to +-180 | Need to adjust lon_right and lon_left for MUSICA!!
lat = spci_ds['lat'].values
lon = spci_ds['lon'].values
Adjustedlons = np.where(lon >= 180, lon - 360, lon)
# Create a DataArray for the adjusted lon
Adjustedlons_da = xr.DataArray(
                                Adjustedlons,  
                                dims=('ncol'),
                                coords={'ncol': spci_ds.ncol.values} 
                                )

# Add the DataArray back to the dataset
spci_ds['Adjustedlons'] = Adjustedlons_da

In [ ]:
spci_ds

In [ ]:
# Quick plot to check

Timei = '2018-07-01T16:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'NYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
geom

In [ ]:
# Create a mask for the dataset
mask = np.zeros(len(lat), dtype=bool)

# Iterate over the geometries and update the mask
points_within_bounds = []
for geom in gdf['geometry']:
    for i in range(len(lat)):
        point = Point(Adjustedlons[i], lat[i])
        if geom.contains(point):
            points_within_bounds.append((Adjustedlons[i], lat[i]))
            mask[i] = True


In [ ]:
points_within_bounds

In [ ]:
# Plot the GeoDataFrame (polygons)
fig, ax = plt.subplots(figsize=(10, 10))
gdf.plot(ax=ax, color='white', edgecolor='tab:blue')

# Plot the points within the boundaries
points_within_bounds = np.array(points_within_bounds)
ax.scatter(points_within_bounds[:, 0], points_within_bounds[:, 1], color='red', marker='o', label='Points within boundaries')

# Customize the plot
ax.set_title('Polygons and Points within NYC Boundaries')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend()

# Show the plot
plt.show()


In [ ]:
# Convert mask to DataArray
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Apply the mask to the dataset
masked_ds = spci_ds.where(mask_da, drop=True)

# Apply the mask to the 'sum' variable
sum_masked = spci_ds['sum'].where(~mask_da, other=np.nan)

In [ ]:
spci_ds

In [ ]:
# Quick plot to check

Timei = '2018-07-01T16:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'NYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = sum_masked.sel(time=Timei).values#*scalefactor
Plot_unit = sum_masked.sel(time=Timei).units#+' ('+formatted_label+')'
longname = sum_masked.sel(time=Timei).long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())

gdf.plot(ax=ax1, color='white', edgecolor='tab:blue')

if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 
    
# finish plotting

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

In [ ]:
# Convert mask to DataArray
mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

# Apply the mask to the dataset
masked_ds = spci_ds.where(mask_da, drop=True)

# Apply the mask to the 'sum' variable
sum_masked = spci_ds['sum'].where(~mask_da, other=np.nan)

spci_ds['sum'] = sum_masked

In [ ]:
spci_ds

In [ ]:
# Quick plot to check

Timei = '2018-07-01T16:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'NYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

## Plot to Check processed dataspci

In [ ]:
spci_filename = f'{P.CESM22_ROOT}/CAMS_withCONUS2017NEI/NYCMasked_MappedSpecies_GLOB_Merged_CAMS_NEI_ne0CONUSne30x8/NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_CH3COCH3_c20231113.nc'
spci_ds = xr.open_dataset(spci_filename)
spci_ds

In [ ]:
spci_ds['sum'].values

In [ ]:
hourlyNEI_diri

In [ ]:
spci_filename = f'{hourlyNEI_diri}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_SVOC_c20231113.nc'
orgspci_ds = xr.open_dataset(spci_filename)
orgspci_ds

In [ ]:
orgspci_ds['sum'].values

In [ ]:
# Quick plot to check

Timei = '2018-07-01T16:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'NYC'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

np.nanmean(Plot_ar)

In [ ]:
np.nanmean(spci_ds.sel(time='2018-07-07T19:00:00.000000000')['sum'].values)-np.nanmean(spci_ds.sel(time='2018-07-01T19:00:00.000000000')['sum'].values)

In [ ]:
np.nanmean(spci_ds.sel(time='2018-07-07T19:00:00.000000000')['sum'].values)-np.nanmean(spci_ds.sel(time='2018-07-01T19:00:00.000000000')['sum'].values)

In [ ]:
# Quick plot to check

Timei = '2018-07-07T10:00:00.000000000'

setmap = 'viridis'

"""ds_mappedMUSICA (regridded to ne0CONUS30x8)"""
PlotRegion = 'CONUS'

# scalefactor = 1e-11
# formatted_label = f"{scalefactor:.0e}" 

# Plot_ar = spci_ds.sel(time='2018-07-07T19:00:00.000000000')['sum'].values-spci_ds.sel(time='2018-07-01T19:00:00.000000000')['sum'].values#*scalefactor
Plot_ar = spci_ds.sel(time=Timei)['sum'].values#*scalefactor
Plot_unit = spci_ds.sel(time=Timei)['sum'].units#+' ('+formatted_label+')'
longname = spci_ds.sel(time=Timei)['sum'].long_name
rangeMax = np.nanmean(Plot_ar) #setvmax*scalefactor

### Which map
fig = plt.figure( figsize=(8,6) ) 
# - ne30x8 regional refinement over CONUS|
ax1 = fig.add_subplot(1,1,1,projection=ccrs.PlateCarree())
if PlotRegion=="CONUS":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-140,-50], lat_range=[15,60],
              grid_line=False, grid_line_lw=0.15 ) 
elif PlotRegion=="Global":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True,grid_line=False, grid_line_lw=0.15 )  
elif PlotRegion=="NYC":
    im = Plot_2D( Plot_ar, scrip_file=SCRIP_CONUS, ax=ax1,
            cmin=0, cmax=rangeMax, cmap=setmap, 
            unit=Plot_unit,
            state=True, lon_range=[-74.5,-73.5], lat_range=[40.4,41],
              grid_line=False, grid_line_lw=0.15 ) 

plt.title(longname+' \n'+Timei[:13], fontsize=16, y=1.02);

np.nanmean(Plot_ar)

# Estimate CONUS Emissions (numbers replaced with zero)

In [ ]:
### Read in NEI emissions data
hourlyNEI_diri = f'{P.CESM22_ROOT}/CAMS_withCONUS2017NEI/MappedSpecies_GLOB_Merged_CAMS_NEI_ne0CONUSne30x8/'
# JulyMeanNEI_diri = '{P.CESM22_ROOT}/CAMS_withCONUS2017NEI/MappedSpecies_GLOB_Merged_CAMS_NEI_ne0CONUSne30x8_JulyMean/'

# get a list of species name I have updated 
MappedSpecies_dir = hourlyNEI_diri
DateOfModify = 'c20231113' #'c20231218';'c20231113'

# Search for files that match the pattern
pattern = '*NEI2017Merged_20181MJuly_CAMSv5.1*'+DateOfModify+'*'
Out_file_list = [file for file in glob.glob(os.path.join(MappedSpecies_dir, pattern)) if fnmatch.fnmatch(os.path.basename(file), pattern)]


In [ ]:
len(Out_file_list)

updated_spcs = []

for fileidx in range(len(Out_file_list)):
    filename = Out_file_list[fileidx].split('/')[-1]
    # Find the starting and ending indexes for the species name
    start_index = filename.find('ne0CONUSne30x8_') + len('ne0CONUSne30x8_')
    end_index = filename.find(DateOfModify)
    # Extract the species name using the start and end indexes
    species_name = filename[start_index:end_index-1]
    # print(species_name) 
    updated_spcs.append(species_name)
    
print(f'Number of updated species: {len(updated_spcs)}')

In [ ]:
### Estimate emissions within NYC boundary

import os
import numpy as np
import xarray as xr
from shapely.geometry import Point

# Define dictionary to store emission sums set to zero
city_emissions = {}

# Read in each file in the format 
# for spc in updated_spcs[:2]:
for spc in updated_spcs:
    # Construct file path
    FileInPath = f'{hourlyNEI_diri}NEI2017Merged_20181MJuly_CAMSv5.1_anthro_2018_ne0CONUSne30x8_{spc}_c20231113.nc'
    
    # Check if the file exists
    if not os.path.exists(FileInPath):
        print(f"{FileInPath} missing.")
        continue

    print(f'Calculating NYC-summed July mean emissions for {spc}')
    print('************************************************************************')

    # Open the dataset
    spci_ds = xr.open_dataset(FileInPath)

    # Adjust longitudes from 0-360 to -180 to 180
    lat = spci_ds['lat'].values
    lon = spci_ds['lon'].values
    Adjustedlons = np.where(lon >= 180, lon - 360, lon)
    Adjustedlons_da = xr.DataArray(Adjustedlons, dims=('ncol'), coords={'ncol': spci_ds.ncol.values})
    spci_ds['Adjustedlons'] = Adjustedlons_da

    # Create a mask based on the boundary geometry
    mask = np.zeros(len(lat), dtype=bool)
    for geom in gdf['geometry']:
        for i in range(len(lat)):
            point = Point(Adjustedlons[i], lat[i])
            if geom.contains(point):
                mask[i] = True

    mask_da = xr.DataArray(mask, dims=["ncol"], coords={"ncol": spci_ds["ncol"]})

    # Initialize dictionary for this city and species if not already created
    if spc not in city_emissions:
        city_emissions[spc] = {}

    # Iterate over variables except those to ignore, apply mask, set to 0, and calculate sum
    excluded_vars = ['lon', 'lat', 'area', 'rrfac', 'date', 'Adjustedlons']
    for var_name in spci_ds.data_vars:
        if var_name not in excluded_vars:
            print(var_name)
            # Set emissions to zero within the boundary
            zeroed_emissions = spci_ds[var_name].where(~mask_da, other=0)
            
            # Calculate sum of emissions that were set to zero
            sum_Cityi_emissions = spci_ds[var_name].where(mask_da).sum().item() # use .item() to get just the number

            # Store the result in the dictionary
            if var_name not in city_emissions[spc]:
                city_emissions[spc][var_name] = sum_Cityi_emissions
                # add unit information
                city_emissions[spc]["unites"] = spci_ds["sum"].units

    print('************************************************************************')



In [ ]:
# Flatten the dictionary to create a DataFrame with separate columns for sum and units
flattened_data = {
    species: {'sum': details['sum'], 'units': details['unites']}
    for species, details in city_emissions.items()
}

# Convert to DataFrame
emissions_df = pd.DataFrame(flattened_data).T

# Format the 'sum' column in scientific notation with 2 decimal places
emissions_df['sum'] = emissions_df['sum'].apply(lambda x: f"{x:.2e}")

emissions_df

In [ ]:
# Export the DataFrame to CSV
csv_path = "./NYCsummed_JulyMean_AnthroEmis.csv"
emissions_df.to_csv(csv_path)

print("Saved to:", csv_path)

#### Draft checking calculations

In [ ]:
city_emissions

In [ ]:
zeroed_emissions.sum()

In [ ]:
var_name = 'sum'

In [ ]:
spci_ds[var_name].sum().item()

In [ ]:
spci_ds[var_name].where(mask_da).sum()

In [ ]:
zeroed_emissions.sum()+spci_ds[var_name].where(mask_da).sum()

In [ ]:
['lon','lat','area','rrfac','date','Adjustedlons']

# Analyze Output Simulations